Imports

In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import sweetviz as sv

---
DuckDB Connection

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent
DB_PATH = PROJECT_ROOT / "database" / "simbox.duckdb"

con = duckdb.connect(str(DB_PATH))

print("DuckDB connected successfully.")

DuckDB connected successfully.


---
### Manual SQL EDA Query

<u>Normal dataset\:</u>

In [3]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT *
FROM cdr_all
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,MSISDN,AnswerTime,DurationInSeconds,LAC,Cell,CallingNumber,CalledNumber,RecordType
0,2250102440302,2022-11-01 00:00:12,23,2032,39965,2250102440302,2250777265980,0
1,2250151569453,2022-11-01 00:00:44,15,2221,35623,2250101362803,2250151569453,1
2,2250151135630,2022-11-01 00:00:15,102,2038,30618,2250709275379,2250151135630,1
3,2250151927977,2022-11-01 00:02:10,18,2082,44702,2250151927977,2250140166230,0
4,2250153338784,2022-11-01 00:02:43,19,2022,44265,2250153338784,2250596386682,0
...,...,...,...,...,...,...,...,...
6849570,2250141621106,2022-11-15 23:50:16,3691,2092,36854,2250142696036,2250141621106,1
6849571,2250140555908,2022-11-15 23:25:32,6073,2081,36717,2250141841714,2250140555908,1
6849572,2250143235427,2022-11-15 23:58:51,3778,2021,33284,2250173663481,2250143235427,1
6849573,2250153968040,2022-11-15 23:03:40,7078,2082,11953,2250143577911,2250153968040,1


User activity (aggregated on MSISDN) 

In [4]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT
    MSISDN,

    COUNT(*) AS total_calls,

    SUM(
        CASE
            WHEN RecordType = 0 THEN 1
            ELSE 0
        END
    ) AS outgoing_calls,

    SUM(
        CASE
            WHEN RecordType = 1 THEN 1
            ELSE 0
        END
    ) AS incoming_calls,

    COUNT(
        DISTINCT CASE
            WHEN RecordType = 1 THEN CallingNumber
        END
    ) AS unique_callers,

    COUNT(
        DISTINCT CASE
            WHEN RecordType = 0 THEN CalledNumber
        END
    ) AS unique_callees,

    SUM(DurationInSeconds) AS total_call_duration,

    AVG(DurationInSeconds) AS avg_call_duration,

    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN RecordType = 0 THEN 1
                    ELSE 0
                END
            ) AS DOUBLE
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN RecordType = 1 THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        4
    ) AS outgoing_incoming_ratio,

    COUNT(DISTINCT LAC) AS unique_LAC,

    COUNT(DISTINCT Cell) AS unique_cells,

    SUM(
        CASE
            WHEN CAST(AnswerTime AS TIME)
                 BETWEEN TIME '00:30:00'
                 AND TIME '05:30:00'
            THEN 1
            ELSE 0
        END
    ) AS night_calls,

    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN CAST(AnswerTime AS TIME)
                         BETWEEN TIME '00:30:00'
                         AND TIME '05:30:00'
                    THEN 1
                    ELSE 0
                END
            ) AS DOUBLE
        )
        /
        NULLIF(COUNT(*), 0),
        4
    ) AS night_activity_ratio

FROM cdr_all

GROUP BY MSISDN

ORDER BY outgoing_incoming_ratio DESC;
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,MSISDN,total_calls,outgoing_calls,incoming_calls,unique_callers,unique_callees,total_call_duration,avg_call_duration,outgoing_incoming_ratio,unique_LAC,unique_cells,night_calls,night_activity_ratio
0,2250161549218,201,199.0,2.0,1,28,26141.0,130.054726,99.5,1,7,3.0,0.0149
1,2250152009283,113,111.0,2.0,2,97,7549.0,66.805310,55.5,1,2,0.0,0.0000
2,2250101076318,50,49.0,1.0,1,28,4998.0,99.960000,49.0,1,6,0.0,0.0000
3,2250171614945,78,76.0,2.0,2,14,31548.0,404.461538,38.0,2,17,0.0,0.0000
4,2250160295066,38,37.0,1.0,1,13,7658.0,201.526316,37.0,2,5,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29795,2250171991178,19,19.0,0.0,0,4,1751.0,92.157895,NaN,1,2,2.0,0.1053
29796,2250160555256,8,8.0,0.0,0,6,353.0,44.125000,NaN,1,2,0.0,0.0000
29797,2250153429518,2,2.0,0.0,0,1,81.0,40.500000,NaN,1,2,0.0,0.0000
29798,2250152348400,1,1.0,0.0,0,1,3.0,3.000000,NaN,1,1,0.0,0.0000


Location dynamics (aggregated on LAC)

In [5]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT LAC, (COUNT(*)/15.0) AS daily_usage, COUNT(DISTINCT Cell) AS unique_cells,
       ROUND(
             COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT Cell), 0),
             2) AS avg_records_per_cell
FROM cdr_all
GROUP BY LAC
ORDER BY daily_usage DESC;
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,LAC,daily_usage,unique_cells,avg_records_per_cell
0,2093,14953.666667,672,333.79
1,151,14926.600000,1050,213.24
2,2221,13656.933333,949,215.86
3,2212,11732.600000,551,319.40
4,2016,10942.266667,467,351.46
...,...,...,...,...
98,1016,399.933333,6,999.83
99,2031,281.466667,9,469.11
100,2011,147.400000,12,184.25
101,1011,12.800000,6,32.00


<u>Fraud dataset\:</u>

In [6]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT *
FROM cdr_fraud
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,MSISDN,AnswerTime,DurationInSeconds,LAC,Cell,CallingNumber,CalledNumber,RecordType
0,2250153685272,2022-11-01 00:00:47,12,1013,10674,2250759021937,2.250154e+12,1
1,2250103889065,2022-11-01 00:02:48,102,2092,31073,2250103889065,2.250557e+12,0
2,2250102016447,2022-11-01 00:02:51,10,2212,30279,2250102016447,2.250144e+12,0
3,2250150385602,2022-11-01 00:03:04,45,2037,33645,2250150385602,2.250150e+12,0
4,2250143415873,2022-11-01 00:03:47,23,2024,43694,2250143415873,2.250555e+12,0
...,...,...,...,...,...,...,...,...
368749,2250151264960,2022-11-15 23:57:34,43,2221,15463,2250151264960,2.250708e+12,0
368750,2250103192301,2022-11-15 23:57:38,21,2092,39872,2250103192301,2.250708e+12,0
368751,2250153425550,2022-11-15 23:58:40,4,1092,17294,2250153425550,2.250798e+12,0
368752,2250153425550,2022-11-15 23:59:31,7,1092,17294,2250153425550,2.250798e+12,0


Fraud user activity (aggregated on MSISDN) 

In [7]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT
    MSISDN,

    COUNT(*) AS total_calls,

    SUM(
        CASE
            WHEN RecordType = 0 THEN 1
            ELSE 0
        END
    ) AS outgoing_calls,

    SUM(
        CASE
            WHEN RecordType = 1 THEN 1
            ELSE 0
        END
    ) AS incoming_calls,

    COUNT(
        DISTINCT CASE
            WHEN RecordType = 1 THEN CallingNumber
        END
    ) AS unique_callers,

    COUNT(
        DISTINCT CASE
            WHEN RecordType = 0 THEN CalledNumber
        END
    ) AS unique_callees,

    SUM(DurationInSeconds) AS total_call_duration,

    AVG(DurationInSeconds) AS avg_call_duration,

    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN RecordType = 0 THEN 1
                    ELSE 0
                END
            ) AS DOUBLE
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN RecordType = 1 THEN 1
                    ELSE 0
                END
            ),
            0
        ),
        4
    ) AS outgoing_incoming_ratio,

    COUNT(DISTINCT LAC) AS unique_LAC,

    COUNT(DISTINCT Cell) AS unique_cells,

    SUM(
        CASE
            WHEN CAST(AnswerTime AS TIME)
                 BETWEEN TIME '00:30:00'
                 AND TIME '05:30:00'
            THEN 1
            ELSE 0
        END
    ) AS night_calls,

    ROUND(
        CAST(
            SUM(
                CASE
                    WHEN CAST(AnswerTime AS TIME)
                         BETWEEN TIME '00:30:00'
                         AND TIME '05:30:00'
                    THEN 1
                    ELSE 0
                END
            ) AS DOUBLE
        )
        /
        NULLIF(COUNT(*), 0),
        4
    ) AS night_activity_ratio

FROM cdr_fraud

GROUP BY MSISDN

ORDER BY outgoing_incoming_ratio DESC;
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,MSISDN,total_calls,outgoing_calls,incoming_calls,unique_callers,unique_callees,total_call_duration,avg_call_duration,outgoing_incoming_ratio,unique_LAC,unique_cells,night_calls,night_activity_ratio
0,2250171095827,681,680.0,1.0,1,558,75253.0,110.503671,680.0,6,22,0.0,0.0000
1,2250151939921,229,228.0,1.0,1,193,36299.0,158.510917,228.0,1,2,4.0,0.0175
2,2250151965022,218,217.0,1.0,1,190,29650.0,136.009174,217.0,1,2,8.0,0.0367
3,2250152009283,113,111.0,2.0,2,97,7549.0,66.805310,55.5,1,2,0.0,0.0000
4,2250141983325,562,552.0,10.0,10,401,55666.0,99.049822,55.2,1,10,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1955,2250151965013,301,301.0,0.0,0,232,30241.0,100.468439,NaN,1,2,17.0,0.0565
1956,2250172263709,185,185.0,0.0,0,158,30582.0,165.308108,NaN,1,2,9.0,0.0486
1957,2250170641436,15,15.0,0.0,0,13,4814.0,320.933333,NaN,1,2,0.0,0.0000
1958,2250172245224,162,162.0,0.0,0,135,22416.0,138.370370,NaN,1,3,9.0,0.0556


Fraud location dynamics (aggregated on LAC)

In [8]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT LAC, (COUNT(*)/15.0) AS daily_usage, COUNT(DISTINCT Cell) AS unique_cells,
       ROUND(
             COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT Cell), 0),
             2) AS avg_records_per_cell
FROM cdr_fraud
GROUP BY LAC
ORDER BY daily_usage DESC;
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,LAC,daily_usage,unique_cells,avg_records_per_cell
0,302,1589.333333,292,81.64
1,151,940.200000,719,19.61
2,2093,680.800000,543,18.81
3,1081,645.933333,165,58.72
4,1022,643.533333,163,59.22
...,...,...,...,...
97,2031,13.066667,9,21.78
98,1095,12.533333,12,15.67
99,1085,6.066667,11,8.27
100,1011,3.266667,3,16.33


Common rows in both datasets

In [9]:
eda_query = """
-- WRITE YOUR SQL QUERY HERE

SELECT * FROM cdr_all
Intersect
SELECT * FROM cdr_fraud;
"""

eda_result = con.execute(eda_query).fetchdf()

eda_result

,MSISDN,AnswerTime,DurationInSeconds,LAC,Cell,CallingNumber,CalledNumber,RecordType
0,2250171477949,2022-11-01 23:54:27,66,282,14063,2250171477949,2.250710e+12,0
1,2250140086962,2022-11-03 10:40:43,446,282,17152,2250140086962,2.250750e+12,0
2,2250143408837,2022-11-04 19:01:40,50,248,1524,2250143408837,2.250173e+12,0
3,2250140135825,2022-11-04 19:22:46,110,151,32404,2250707186331,2.250140e+12,1
4,2250141454798,2022-11-04 20:00:25,20,2221,31757,2250141454798,2.250141e+12,0
...,...,...,...,...,...,...,...,...
5287,2250161340110,2022-11-11 09:31:36,86,1081,13124,2250161340110,2.250546e+12,0
5288,2250102723098,2022-11-13 09:51:37,61,267,5624,2250102723098,2.250141e+12,0
5289,2250161340110,2022-11-13 12:05:33,40,1081,4002,2250161340110,2.250587e+12,0
5290,2250140086962,2022-11-15 14:29:02,43,282,17152,2250140086962,2.250710e+12,0


---
Close database connection

In [10]:
con.close()